# 3. XGBoost Model with Calibrated Probabilities

**The model.** XGBoost on the 130 features in `model_dataframe_2009_to_2026.pkl`, with the settings Optuna chose (tuned on 2009 to 2014).

**The test.** Walk forward by year: the reports of each two test years are scored by a model trained on reports filed 2 or more years before, so their answers were already known.

**Calibration.** The raw XGBoost score is not a probability (the class weights push it up). Two ways to turn it into one are tried, both fitted only on the training part of each step:
- **Isotonic**: a step function from score to the share of 1s.
- **Platt**: a logistic curve on the score.

The one with the lower log loss on 2016 to 2021 is kept. Log loss punishes a confident wrong probability much more than the Brier score, which matters when only about 1% of the reports are 1s.

In [1]:
import os
import time

import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from xgboost import XGBClassifier

everything = pd.read_pickle("model_dataframe_2009_to_2026.pkl")
info = ["adsh", "cik", "name", "sic", "form", "fp", "ddate", "filed", "filed_date", "bk_date", "days_to_bankruptcy",
        "Bankruptcy", "already_bankrupt", "label_known"]
features = [c for c in everything.columns if c not in info]
print(len(features), "features")

#only reports whose answer is known, and not from companies already in bankruptcy
df = everything[everything["label_known"] & ~everything["already_bankrupt"]].reset_index(drop=True)
df["year"] = df["filed_date"].dt.year

XGB_PARAMS = {"n_estimators": 1400, "learning_rate": 0.030435151413863663, "max_depth": 8, "min_child_weight": 2.649496317267566,
              "subsample": 0.8794818753946948, "colsample_bytree": 0.6094118179765666, "gamma": 0.43319316787210643,
              "reg_alpha": 0.0015962113607134825, "reg_lambda": 5.696284007633162, "max_delta_step": 1,
              "scale_pos_weight": 6.61283698692129}


def xgb_model():
    return XGBClassifier(**XGB_PARAMS, tree_method="hist", max_bin=128, eval_metric="aucpr", n_jobs=8, random_state=42)

130 features


## Walk Forward with Calibration

For each two test years the model is trained on the reports filed 2 or more years before. To fit the calibration without using the test years, the training part is split into 4 groups of companies, and each group is scored by a model trained on the other 3 (out of fold). Isotonic and Platt are fitted on those out of fold scores and applied to the test scores.

In [2]:
def out_of_fold(data):
    #each training report scored by a model that didn't see its company
    scores = np.zeros(len(data))
    folds = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=7)
    for fit_index, val_index in folds.split(data, data["Bankruptcy"], groups=data["cik"]):
        model = xgb_model().fit(data.iloc[fit_index][features], data.iloc[fit_index]["Bankruptcy"])
        scores[val_index] = model.predict_proba(data.iloc[val_index][features])[:, 1]
    return scores


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)


def fit_calibration(raw, y):
    isotonic = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(raw, y)
    platt = LogisticRegression(C=1e6).fit(logit(raw), y)
    return isotonic, platt


start = time.time()
parts = []
for year in range(2016, 2026, 2):
    fit = df[df["year"] <= year - 2].reset_index(drop=True)
    test = df[df["year"].isin([year, year + 1])]
    raw_test = xgb_model().fit(fit[features], fit["Bankruptcy"]).predict_proba(test[features])[:, 1]
    isotonic, platt = fit_calibration(out_of_fold(fit), fit["Bankruptcy"])
    parts.append(pd.DataFrame({"year": test["year"].values, "raw": raw_test, "isotonic": isotonic.predict(raw_test),
                               "platt": platt.predict_proba(logit(raw_test))[:, 1]}, index=test.index))
    print(year, year + 1, "done", round((time.time() - start) / 60, 1), "minutes", flush=True)
scores = pd.concat(parts)

2016 2017 done 0.5 minutes


2018 2019 done 1.4 minutes


2020 2021 done 2.5 minutes


2022 2023 done 4.0 minutes


2024 2025 done 5.6 minutes


## Which Calibration

Brier score and log loss (lower is better) measure how close the probabilities are to what happened. PR-AUC and ROC-AUC only depend on the order, so calibration should barely change them.

In [3]:
def scores_table(years):
    part = scores[scores["year"].isin(years)]
    y = df.loc[part.index, "Bankruptcy"]
    rows = []
    for name in ["raw", "isotonic", "platt"]:
        p = part[name].clip(1e-6, 1 - 1e-6)
        rows.append({"probability": name, "Brier": brier_score_loss(y, p), "log loss": log_loss(y, p),
                     "PR-AUC": average_precision_score(y, p), "ROC-AUC": roc_auc_score(y, p),
                     "average probability": p.mean(), "share that went bankrupt": y.mean()})
    return pd.DataFrame(rows).set_index("probability")


check = scores_table(range(2016, 2022))
CALIBRATION = check.loc[["isotonic", "platt"], "log loss"].idxmin()
print("kept:", CALIBRATION)
print("2016 to 2021")
print(check.round(4))
print("\n2023 to 2025")
print(scores_table(range(2023, 2026)).round(4))

kept: platt
2016 to 2021
              Brier  log loss  PR-AUC  ROC-AUC  average probability  \
probability                                                           
raw          0.0068    0.0380  0.1569   0.8844               0.0038   
isotonic     0.0067    0.0331  0.1472   0.8848               0.0077   
platt        0.0067    0.0328  0.1581   0.8848               0.0078   

             share that went bankrupt  
probability                            
raw                            0.0073  
isotonic                       0.0073  
platt                          0.0073  

2023 to 2025
              Brier  log loss  PR-AUC  ROC-AUC  average probability  \
probability                                                           
raw          0.0088     0.045  0.1479   0.9106               0.0045   
isotonic     0.0086     0.039  0.1438   0.9094               0.0091   
platt        0.0086     0.039  0.1481   0.9112               0.0090   

             share that went bankrupt  
probabili

In [4]:
#by probability band: the average probability against the share that really went bankrupt (2016 to 2025)
y = df.loc[scores.index, "Bankruptcy"]
bands = pd.cut(scores[CALIBRATION], [0, 0.01, 0.05, 0.1, 0.2, 0.5, 1], include_lowest=True)
pd.DataFrame({"band": bands, "probability": scores[CALIBRATION], "raw": scores["raw"], "went bankrupt": y}).groupby(
    "band", observed=True).agg(reports=("went bankrupt", "size"), average_probability=("probability", "mean"),
                               share_that_went_bankrupt=("went bankrupt", "mean"), average_raw_score=("raw", "mean")).round(4)

,reports,average_probability,share_that_went_bankrupt,average_raw_score
band,,,,
"(-0.001, 0.01]",200079,0.0019,0.0022,0.0002
"(0.01, 0.05]",24278,0.0213,0.0269,0.0054
"(0.05, 0.1]",3999,0.0696,0.0588,0.0308
"(0.1, 0.2]",1946,0.1376,0.1023,0.0895
"(0.2, 0.5]",967,0.2993,0.2368,0.2989
"(0.5, 1.0]",166,0.6263,0.5181,0.7512


## Probabilities for the Investigator Agent

The agent works on reports filed from 2023 on, so the file has:
- **2023 to 2025 reports with a known answer**: the calibrated probability from the walk forward above (from a model that never saw them).
- **Newer reports without an answer yet**: a model trained on every report with a known answer, calibrated the same way on its out of fold scores.

Saved as `agent/shared/risk_scores_calibrated.csv`, with the same columns as `risk_scores.csv` plus the raw score.

In [5]:
columns = ["adsh", "cik", "name", "form", "ddate", "filed"]
known = df[df["year"] >= 2023]
known_scores = known[columns].assign(probability=scores.loc[known.index, CALIBRATION], raw_score=scores.loc[known.index, "raw"],
                                     scored_by="walk forward, label known")

new_reports = everything[~everything["label_known"] & ~everything["already_bankrupt"]]
start = time.time()
model_all = xgb_model().fit(df[features], df["Bankruptcy"])
isotonic_all, platt_all = fit_calibration(out_of_fold(df), df["Bankruptcy"])
raw_new = model_all.predict_proba(new_reports[features])[:, 1]
calibrated_new = isotonic_all.predict(raw_new) if CALIBRATION == "isotonic" else platt_all.predict_proba(logit(raw_new))[:, 1]
new_scores = new_reports[columns].assign(probability=calibrated_new, raw_score=raw_new, scored_by="all labeled reports, label not known yet")
print("took", round((time.time() - start) / 60, 1), "minutes")

risk_scores = pd.concat([known_scores, new_scores]).sort_values("probability", ascending=False)
os.makedirs("agent/shared", exist_ok=True)
risk_scores.to_csv("agent/shared/risk_scores_calibrated.csv", index=False)
print(len(risk_scores), "reports saved")
print(risk_scores["scored_by"].value_counts())

took 2.1 minutes


85753 reports saved
scored_by
walk forward, label known                   63767
all labeled reports, label not known yet    21986
Name: count, dtype: int64


In [6]:
#the highest probabilities among the newest reports (no answer yet), for the agent to start with
risk_scores[risk_scores["scored_by"].str.startswith("all labeled")].head(10)[["name", "form", "ddate", "filed", "probability", "raw_score"]].round(4)

,name,form,ddate,filed,probability,raw_score
375116,SLEEP NUMBER CORP,10-K,20251231,20260312,0.8522,0.9580
368203,IROBOT CORP,10-Q,20250930,20251106,0.8395,0.9517
376297,"HYDROFARM HOLDINGS GROUP, INC.",10-K,20251231,20260327,0.8071,0.9338
369792,"FAT BRANDS, INC",10-Q,20250930,20251112,0.7738,0.9125
370854,ZYNEX INC,10-Q,20250930,20251117,0.7716,0.9110
367938,"QVC GROUP, INC.",10-Q,20250930,20251105,0.7591,0.9021
366811,CUMULUS MEDIA INC,10-Q,20250930,20251030,0.7559,0.8998
380352,SLEEP NUMBER CORP,10-Q,20260331,20260512,0.7474,0.8935
385160,NEW FORTRESS ENERGY INC.,10-Q,20260630,20260806,0.7334,0.8826
369793,TWIN HOSPITALITY GROUP INC.,10-Q,20250930,20251112,0.7192,0.8710
